In [ ]:

import os
import math
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ============================================================
# STATEFUL LOW-LATENCY ADAPTIVE SCHEDULER EVALUATION
# - Traverses the ENTIRE dense sequence in temporal order
# - First frame: mandatory FIRE
# - Following frames: the scheduler estimates dev_pred
# - FIRE if dev_pred > ADAPTIVE_TAU
# - HOLD if dev_pred <= ADAPTIVE_TAU
# - Evaluates ONLY annotated frames, following the same DFF/Clockwork philosophy
# - Exports an Excel file with per-frame data, summary, and confusion matrix
# ============================================================

# ======================== EDIT ===============================
os.chdir('/mmsegmentation')

CONFIG = '/mmsegmentation/zmax_configs/04_test_adaptive_scheduler_stateful.py'

# If CKPT=None, it tries to resolve it from cfg.CKPT_PATH or cfg.load_from.
# You can set it manually, for example:
CKPT   = '/mmsegmentation/work_dirs/bisenet_adaptive_scheduler_from_pretrain02/best_cls_acc_cls_top1_iter_16200.pth'
DEVICE = 'cuda:0'

SEQ_ROOT     = '/0_secuencias_ann/1a_secuencia'
IMG_DIR      = '/0_secuencia_paravideo/1a_secuencia2_paravideo/'
MASK_DIR     = f'{SEQ_ROOT}/ann'
CLS_ANN_PATH = f'{SEQ_ROOT}/cls_labels_train.txt'

# Excel output
ADAPTIVE_TAU = 0.03
MAX_HOLD_FRAMES = None       # None: scheduler-only decision; integer: force FIRE every N consecutive HOLD frames
FIRST_FRAME_FIRE = True
OUTPUT_XLSX  = f'/mmsegmentation/output/eval_stateful_adaptive_scheduler_tau{ADAPTIVE_TAU:.3f}_fullseq_1a.xlsx'

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp'}
MASK_EXT = '.png'
NUM_SEG_CLASSES = 2
IGNORE_INDEX = 255
GT_MASK_BINARY = True

# The base model was trained with internal labels in Spanish.
# If your cls_head now returns LEFT/STRAIGHT/RIGHT, change this list.
MODEL_CLS_NAMES = ['izquierda', 'recta', 'derecha']
GT_LABEL_MAP = {
    'LEFT': 'izquierda',
    'STRAIGHT': 'recta',
    'RIGHT': 'derecha',
    'IZQUIERDA': 'izquierda',
    'RECTA': 'recta',
    'DERECHA': 'derecha',
    'izquierda': 'izquierda',
    'recta': 'recta',
    'derecha': 'derecha',
}

TIMESTAMP_MATCH_TOL = 1e-4
EVALUATE_ONLY_ANNOTATED_TIMESTAMPS = True

PREPROCESS_USE_PINNED = True
PRED_MASK_DTYPE = np.uint8
USE_AUTOCAST = False
MEASURE_GPU_TIME = True
# ============================================================

import os.path as osp
from glob import glob
from mmengine.config import Config
from mmengine.runner import load_checkpoint
from mmengine.registry import init_default_scope
from mmseg.registry import MODELS
from mmseg.utils import register_all_modules


def ensure_dir_for_file(path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)


def parse_timestamp_from_name(path: str) -> Optional[float]:
    stem = Path(path).stem
    try:
        return float(stem)
    except Exception:
        return None


def sorted_image_paths(img_dir: str) -> List[str]:
    paths = [str(p) for p in Path(img_dir).iterdir() if p.suffix.lower() in IMG_EXTS]
    def _key(p: str):
        t = parse_timestamp_from_name(p)
        return (0, t) if t is not None else (1, Path(p).name)
    return sorted(paths, key=_key)


def normalize_name_key(name: str) -> Tuple[str, str]:
    base = os.path.basename(name)
    stem = os.path.splitext(base)[0]
    return base, stem


def normalize_gt_label(raw_label: str) -> str:
    key = str(raw_label).strip()
    key_upper = key.upper()
    if key in GT_LABEL_MAP:
        return GT_LABEL_MAP[key]
    if key_upper in GT_LABEL_MAP:
        return GT_LABEL_MAP[key_upper]
    raise KeyError(f'La etiqueta GT "{raw_label}" no existe en GT_LABEL_MAP.')


def ts_key(ts: float) -> int:
    return int(round(float(ts) / TIMESTAMP_MATCH_TOL))


def parse_cls_annotations(path: str):
    ann_by_name = {}
    ann_by_ts = {}

    name2id = {n: i for i, n in enumerate(MODEL_CLS_NAMES)}

    with open(path, 'r', encoding='utf-8') as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            parts = line.split(maxsplit=1)
            if len(parts) != 2:
                raise ValueError(f'Linea invalida {ln} en {path}: "{line}"')
            img_name_raw, raw_label = parts
            gt_name = normalize_gt_label(raw_label)
            if gt_name not in name2id:
                raise KeyError(f'GT normalizado {gt_name} no existe en MODEL_CLS_NAMES={MODEL_CLS_NAMES}')
            gt_idx = name2id[gt_name]

            base, stem = normalize_name_key(img_name_raw)
            ts = None
            try:
                ts = float(stem)
            except Exception:
                ts = None

            item = {
                'image_name': base,
                'image_name_raw': img_name_raw,
                'stem': stem,
                'timestamp_s': ts,
                'gt_cls_name': gt_name,
                'gt_cls_idx': gt_idx,
                'raw_label': raw_label,
            }
            ann_by_name[base] = item
            ann_by_name[stem] = item
            if ts is not None:
                ann_by_ts[ts_key(ts)] = item
    return ann_by_name, ann_by_ts


def load_gt_mask(path: str) -> np.ndarray:
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        raise FileNotFoundError(f'No se pudo leer mascara: {path}')
    if GT_MASK_BINARY:
        m = (m > 0).astype(np.uint8)
    return m


def fast_hist(gt: np.ndarray, pred: np.ndarray, num_classes: int, ignore_index: int = 255) -> np.ndarray:
    gt = gt.astype(np.int64, copy=False)
    pred = pred.astype(np.int64, copy=False)
    mask = (gt != ignore_index) & (gt >= 0) & (gt < num_classes)
    hist = np.bincount(
        num_classes * gt[mask].ravel() + pred[mask].ravel(),
        minlength=num_classes ** 2
    ).reshape(num_classes, num_classes)
    return hist


def ious_from_hist(hist: np.ndarray) -> np.ndarray:
    denom = hist.sum(1) + hist.sum(0) - hist.diagonal()
    with np.errstate(divide='ignore', invalid='ignore'):
        ious = hist.diagonal() / denom
    return ious


def safe_nanmean(arr) -> float:
    arr = np.asarray(arr, dtype=np.float64)
    return float(np.nanmean(arr)) if arr.size else math.nan


def ensure_cm_size(cm: np.ndarray, size: int) -> np.ndarray:
    if cm.shape == (size, size):
        return cm
    out = np.zeros((size, size), dtype=np.int64)
    h = min(size, cm.shape[0]); w = min(size, cm.shape[1])
    out[:h, :w] = cm[:h, :w]
    return out


def add_cls_pair(cm: np.ndarray, gt_idx: int, pred_idx: int) -> np.ndarray:
    cm = ensure_cm_size(cm, len(MODEL_CLS_NAMES))
    if 0 <= gt_idx < cm.shape[0] and 0 <= pred_idx < cm.shape[1]:
        cm[gt_idx, pred_idx] += 1
    return cm


def cls_top1_from_cm(cm: np.ndarray) -> float:
    total = cm.sum()
    return float(cm.diagonal().sum() / total) if total > 0 else math.nan


def find_mask_path(mask_dir: str, img_name: str) -> str:
    cand1 = os.path.join(mask_dir, img_name)
    if os.path.exists(cand1):
        return cand1
    stem = Path(img_name).stem
    cand2 = os.path.join(mask_dir, stem + MASK_EXT)
    if os.path.exists(cand2):
        return cand2
    raise FileNotFoundError(f'No se encontro mascara para {img_name} en {mask_dir}')


def _disable_pretrained_and_init_cfg(obj):
    """Avoids loading the model zoo during testing. The final checkpoint takes precedence."""
    if isinstance(obj, dict):
        obj.pop('pretrained', None)
        if 'init_cfg' in obj:
            obj['init_cfg'] = None
        for v in obj.values():
            _disable_pretrained_and_init_cfg(v)
    elif isinstance(obj, list):
        for v in obj:
            _disable_pretrained_and_init_cfg(v)


def _resolve_ckpt_path(config_path: str, ckpt_path: Optional[str]) -> str:
    if ckpt_path is not None and os.path.isfile(ckpt_path):
        return ckpt_path

    cfg = Config.fromfile(config_path)
    for key in ['CKPT_PATH', 'load_from']:
        try:
            cand = cfg.get(key, None)
        except Exception:
            cand = None
        if cand and os.path.isfile(str(cand)):
            return str(cand)

    work_dir = cfg.get('work_dir', None)
    if work_dir:
        cand_patterns = [
            'best_seg_mIoU_iter_*.pth',
            'best_cls_acc_cls_top1_iter_*.pth',
            'latest.pth',
            'iter_*.pth',
        ]
        cands = []
        for pat in cand_patterns:
            cands.extend(glob(os.path.join(work_dir, pat)))
        cands = [p for p in cands if os.path.isfile(p)]
        if cands:
            # Prefer latest if it exists; otherwise use the newest by mtime.
            latest = [p for p in cands if os.path.basename(p) == 'latest.pth']
            if latest:
                return latest[0]
            return sorted(cands, key=os.path.getmtime, reverse=True)[0]

    raise FileNotFoundError(
        'No pude resolver checkpoint. Define CKPT manualmente o cfg.CKPT_PATH/load_from válido.'
    )


class AdaptiveSchedulerFullSeqInferencer:
    """Stateful inference for BiSeNetAdaptiveVideoSegmentor.

    FIRE:
        Spatial Path + Context Path + FFM; updates the cache.
    HOLD:
        Current Spatial Path + feature_propagation(C_key, S_key, S_t) + FFM.
    Scheduler:
        dev_pred = keyframe_selector(S_key, S_t)
        FIRE if dev_pred > tau.
    """

    def __init__(self, model, cfg, tau=0.02, max_hold_frames=None, first_frame_fire=True):
        self.model = model
        self.model.eval()
        self.cfg = cfg
        self.device = next(self.model.parameters()).device

        self.tau = float(tau)
        self.max_hold_frames = None if max_hold_frames is None else int(max_hold_frames)
        self.first_frame_fire = bool(first_frame_fire)

        self.input_w = 512
        self.input_h = 512
        try:
            st_cfg = cfg.get('adaptive_stateful_cfg', {})
            size = st_cfg.get('input_size', (512, 512))
            self.input_w = int(size[0])
            self.input_h = int(size[1])
        except Exception:
            pass

        cfg_dp = cfg.model.get('data_preprocessor', {})
        mean = np.array(cfg_dp.get('mean', [123.675, 116.28, 103.53]), dtype=np.float32)
        std  = np.array(cfg_dp.get('std',  [58.395, 57.12, 57.375]), dtype=np.float32)
        self.bgr_to_rgb = bool(cfg_dp.get('bgr_to_rgb', True))

        self.mean = torch.tensor(mean, device=self.device, dtype=torch.float32).view(1, 3, 1, 1)
        self.std  = torch.tensor(std,  device=self.device, dtype=torch.float32).view(1, 3, 1, 1)

        self.use_cuda = (self.device.type == 'cuda')
        self.use_pinned = bool(PREPROCESS_USE_PINNED and self.use_cuda)

        self._resize_hwc = np.empty((self.input_h, self.input_w, 3), dtype=np.uint8)
        self._cpu_hwc_t = None
        self._cpu_hwc_np = None
        self._gpu_hwc_u8 = None
        self._gpu_chw_f32 = None

        if self.use_pinned:
            self._cpu_hwc_t = torch.empty((1, self.input_h, self.input_w, 3), dtype=torch.uint8, pin_memory=True)
            self._cpu_hwc_np = self._cpu_hwc_t[0].numpy()
            self._gpu_hwc_u8 = torch.empty((1, self.input_h, self.input_w, 3), dtype=torch.uint8, device=self.device)
            self._gpu_chw_f32 = torch.empty((1, 3, self.input_h, self.input_w), dtype=torch.float32, device=self.device)

        self.reset_state()

    def reset_state(self):
        self.cache = None
        self.frame_counter = 0
        self.hold_count_since_fire = 0
        self.last_phase = 'INIT'
        self.last_dev_pred = np.nan
        self.last_fire_reason = 'RESET'

    def _preprocess(self, img_bgr: np.ndarray) -> torch.Tensor:
        if (img_bgr.shape[1], img_bgr.shape[0]) != (self.input_w, self.input_h):
            cv2.resize(img_bgr, (self.input_w, self.input_h), dst=self._resize_hwc, interpolation=cv2.INTER_LINEAR)
            img = self._resize_hwc
        else:
            img = img_bgr

        if self.bgr_to_rgb:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.use_pinned:
            np.copyto(self._cpu_hwc_np, img)
            self._gpu_hwc_u8.copy_(self._cpu_hwc_t, non_blocking=True)
            chw_u8 = self._gpu_hwc_u8.permute(0, 3, 1, 2)
            self._gpu_chw_f32.copy_(chw_u8)
            self._gpu_chw_f32.sub_(self.mean).div_(self.std)
            return self._gpu_chw_f32

        chw = np.ascontiguousarray(img.transpose(2, 0, 1))
        tensor = torch.from_numpy(chw).unsqueeze(0)
        if self.use_cuda:
            tensor = tensor.to(self.device, non_blocking=True)
        tensor = tensor.float()
        tensor = (tensor - self.mean) / self.std
        return tensor

    def _decode_from_fuse(self, x_fuse, context8_like, context16_like, spatial_like):
        feats = (x_fuse, context8_like, context16_like, spatial_like)
        seg_logits = self.model.decode_head.forward(feats)
        if seg_logits.shape[-2:] != (self.input_h, self.input_w):
            seg_logits = F.interpolate(
                seg_logits,
                size=(self.input_h, self.input_w),
                mode='bilinear',
                align_corners=False,
            )
        pred_mask = seg_logits.argmax(dim=1)[0].detach().to(torch.uint8).cpu().numpy()

        cls_idx = None
        if getattr(self.model, 'cls_head', None) is not None:
            cls_logits = self.model.cls_head.forward(x_fuse)
            cls_idx = int(torch.argmax(cls_logits, dim=1)[0].detach().cpu().item())
        if cls_idx is None:
            raise RuntimeError('No se encontró predicción de clasificación en cls_head.')
        return pred_mask, cls_idx

    def _run_full_fire(self, inputs, spatial_cur=None):
        x_context8, x_context16 = self.model.backbone.context_path(inputs)
        if spatial_cur is None:
            spatial_cur = self.model.backbone.spatial_path(inputs)
        x_fuse = self.model.backbone.ffm(spatial_cur, x_context8)

        self.cache = dict(
            spatial_key=spatial_cur.detach(),
            context8_key=x_context8.detach(),
            context16_key=x_context16.detach(),
        )

        pred_mask, cls_idx = self._decode_from_fuse(x_fuse, x_context8, x_context16, spatial_cur)
        return pred_mask, cls_idx

    def _run_hold_prop(self, spatial_cur):
        context8_prop = self.model.feature_propagation(
            context_key=self.cache['context8_key'],
            spatial_key=self.cache['spatial_key'],
            spatial_cur=spatial_cur,
        )
        context16_like = self.cache.get('context16_key', context8_prop)
        x_fuse = self.model.backbone.ffm(spatial_cur, context8_prop)
        pred_mask, cls_idx = self._decode_from_fuse(x_fuse, context8_prop, context16_like, spatial_cur)
        return pred_mask, cls_idx

    @torch.inference_mode()
    def __call__(self, img_bgr: np.ndarray):
        if self.use_cuda and MEASURE_GPU_TIME:
            torch.cuda.synchronize()
            ev_start = torch.cuda.Event(enable_timing=True)
            ev_end = torch.cuda.Event(enable_timing=True)
            ev_start.record()
        else:
            ev_start = ev_end = None

        inputs = self._preprocess(img_bgr)

        dev_pred = np.nan
        fire_reason = ''

        if self.cache is None:
            do_fire = True
            fire_reason = 'FIRST_OR_EMPTY_CACHE'
            spatial_cur = None
        else:
            spatial_cur = self.model.backbone.spatial_path(inputs)
            if getattr(self.model, 'keyframe_selector', None) is None:
                dev_tensor = torch.tensor([1.0], device=self.device)
            else:
                dev_tensor = self.model.keyframe_selector(self.cache['spatial_key'], spatial_cur)
            dev_pred = float(dev_tensor.detach().cpu().reshape(-1)[0].item())

            if dev_pred > self.tau:
                do_fire = True
                fire_reason = 'DEV_GT_TAU'
            elif self.max_hold_frames is not None and self.hold_count_since_fire >= self.max_hold_frames:
                do_fire = True
                fire_reason = 'MAX_HOLD'
            else:
                do_fire = False
                fire_reason = 'DEV_LE_TAU'

        if do_fire:
            pred_mask, cls_idx = self._run_full_fire(inputs, spatial_cur=spatial_cur if 'spatial_cur' in locals() else None)
            self.hold_count_since_fire = 0
            phase = 'FIRE'
        else:
            pred_mask, cls_idx = self._run_hold_prop(spatial_cur)
            self.hold_count_since_fire += 1
            phase = 'HOLD'

        self.last_phase = phase
        self.last_dev_pred = dev_pred
        self.last_fire_reason = fire_reason
        self.frame_counter += 1

        gpu_ms = np.nan
        if self.use_cuda and MEASURE_GPU_TIME:
            ev_end.record()
            torch.cuda.synchronize()
            gpu_ms = float(ev_start.elapsed_time(ev_end))

        pred_cls_name = MODEL_CLS_NAMES[int(cls_idx)] if 0 <= int(cls_idx) < len(MODEL_CLS_NAMES) else str(cls_idx)
        return pred_mask, int(cls_idx), pred_cls_name, phase, dev_pred, fire_reason, gpu_ms


def prepare_model():
    register_all_modules(init_default_scope=False)
    resolved_ckpt = _resolve_ckpt_path(CONFIG, CKPT)

    cfg = Config.fromfile(CONFIG)
    _disable_pretrained_and_init_cfg(cfg._cfg_dict)

    if 'model' in cfg:
        cfg.model.train_cfg = None

    init_default_scope(cfg.get('default_scope', 'mmseg'))
    model = MODELS.build(cfg.model)
    load_checkpoint(model, resolved_ckpt, map_location='cpu')
    model.cfg = cfg
    model.to(DEVICE)
    model.eval()

    infer = AdaptiveSchedulerFullSeqInferencer(
        model,
        cfg,
        tau=ADAPTIVE_TAU,
        max_hold_frames=MAX_HOLD_FRAMES,
        first_frame_fire=FIRST_FRAME_FIRE,
    )
    return model, infer, resolved_ckpt, cfg


def evaluate_sequence():
    img_paths = sorted_image_paths(IMG_DIR)
    if len(img_paths) == 0:
        raise RuntimeError(f'No se encontraron imagenes en {IMG_DIR}')

    ann_by_name, ann_by_ts = parse_cls_annotations(CLS_ANN_PATH)
    _, infer, resolved_ckpt, cfg = prepare_model()
    infer.reset_state()

    results = []
    cls_cm = np.zeros((len(MODEL_CLS_NAMES), len(MODEL_CLS_NAMES)), dtype=np.int64)
    seg_hist_total = np.zeros((NUM_SEG_CLASSES, NUM_SEG_CLASSES), dtype=np.int64)

    t0 = parse_timestamp_from_name(img_paths[0])

    fire_count = 0
    hold_count = 0
    last_fire_idx = None

    for idx, img_path in enumerate(tqdm(img_paths, desc='Evaluando ADAPTIVE SCHEDULER LOW-LATENCY full sequence')):
        img_name = os.path.basename(img_path)
        timestamp = parse_timestamp_from_name(img_path)
        time_rel_s = (timestamp - t0) if (timestamp is not None and t0 is not None) else float(idx)

        img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(f'No se pudo leer imagen: {img_path}')

        pred_mask_small, pred_cls_idx, pred_cls_name, mode_used, dev_pred, fire_reason, gpu_ms = infer(img_bgr)

        if mode_used == 'FIRE':
            fire_count += 1
            last_fire_idx = idx
        else:
            hold_count += 1

        gt_info = None
        if timestamp is not None:
            gt_info = ann_by_ts.get(ts_key(timestamp), None)
        evaluated = bool(gt_info is not None) if EVALUATE_ONLY_ANNOTATED_TIMESTAMPS else True

        row = {
            'frame_idx': int(idx),
            'image_name': img_name,
            'timestamp_s': float(timestamp) if timestamp is not None else np.nan,
            'time_rel_s': float(time_rel_s),
            'mode_used': mode_used,
            'is_fire': int(mode_used == 'FIRE'),
            'is_hold': int(mode_used == 'HOLD'),
            'last_fire_frame_idx': int(last_fire_idx) if last_fire_idx is not None else np.nan,
            'adaptive_tau': float(ADAPTIVE_TAU),
            'dev_pred': float(dev_pred) if not np.isnan(dev_pred) else np.nan,
            'fire_reason': fire_reason,
            'max_hold_frames': MAX_HOLD_FRAMES if MAX_HOLD_FRAMES is not None else 'None',
            'gpu_ms': float(gpu_ms) if not np.isnan(gpu_ms) else np.nan,
            'evaluated_with_gt': int(evaluated),
            'pred_cls_idx': int(pred_cls_idx),
            'pred_cls_name': pred_cls_name,
            'gt_cls_idx': np.nan,
            'gt_cls_name': None,
            'cls_correct': np.nan,
            'seg_mIoU': np.nan,
            'seg_IoU_background': np.nan,
            'seg_IoU_road': np.nan,
            'mask_path': None,
            'gt_label_raw': None,
        }

        if evaluated:
            if gt_info is None:
                raise RuntimeError(f'El frame {img_name} quedó marcado para evaluar, pero no se encontró GT de clasificación.')

            mask_path = find_mask_path(MASK_DIR, gt_info['image_name'])
            gt_mask = load_gt_mask(mask_path)
            pred_mask = cv2.resize(
                pred_mask_small.astype(np.uint8),
                (gt_mask.shape[1], gt_mask.shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )

            hist = fast_hist(gt_mask, pred_mask, NUM_SEG_CLASSES, IGNORE_INDEX)
            seg_hist_total += hist
            ious = ious_from_hist(hist)
            miou = float(np.nanmean(ious))
            iou_bg = float(ious[0]) if len(ious) > 0 else math.nan
            iou_road = float(ious[1]) if len(ious) > 1 else math.nan

            gt_cls_idx = int(gt_info['gt_cls_idx'])
            gt_cls_name = str(gt_info['gt_cls_name'])
            cls_cm = add_cls_pair(cls_cm, gt_cls_idx, pred_cls_idx)

            row.update({
                'gt_cls_idx': gt_cls_idx,
                'gt_cls_name': gt_cls_name,
                'cls_correct': int(gt_cls_idx == int(pred_cls_idx)),
                'seg_mIoU': miou,
                'seg_IoU_background': iou_bg,
                'seg_IoU_road': iou_road,
                'mask_path': mask_path,
                'gt_label_raw': gt_info['raw_label'],
            })

        results.append(row)

    df = pd.DataFrame(results)
    eval_df = df[df['evaluated_with_gt'] == 1].copy()
    seg_ious_total = ious_from_hist(seg_hist_total)

    dev_all = df['dev_pred'].dropna().astype(float).to_numpy() if 'dev_pred' in df else np.array([])
    gpu_all = df['gpu_ms'].dropna().astype(float).to_numpy() if 'gpu_ms' in df else np.array([])

    summary = {
        'method': 'bisenet_adaptive_scheduler_stateful_low_latency',
        'n_all_frames': int(len(df)),
        'n_evaluated_frames': int(len(eval_df)),
        'fire_count_all': int((df['mode_used'] == 'FIRE').sum()),
        'hold_count_all': int((df['mode_used'] == 'HOLD').sum()),
        'fire_ratio_all': float((df['mode_used'] == 'FIRE').mean()) if len(df) else math.nan,
        'fire_count_evaluated': int(((eval_df['mode_used'] == 'FIRE')).sum()),
        'hold_count_evaluated': int(((eval_df['mode_used'] == 'HOLD')).sum()),
        'adaptive_tau': float(ADAPTIVE_TAU),
        'max_hold_frames': MAX_HOLD_FRAMES if MAX_HOLD_FRAMES is not None else 'None',
        'mean_dev_pred_all_nonfirst': float(dev_all.mean()) if dev_all.size else math.nan,
        'median_dev_pred_all_nonfirst': float(np.median(dev_all)) if dev_all.size else math.nan,
        'p95_dev_pred_all_nonfirst': float(np.percentile(dev_all, 95)) if dev_all.size else math.nan,
        'mean_gpu_ms_all': float(gpu_all.mean()) if gpu_all.size else math.nan,
        'fps_from_mean_gpu_ms_all': float(1000.0 / gpu_all.mean()) if gpu_all.size and gpu_all.mean() > 0 else math.nan,
        'mean_frame_mIoU_eval_only': safe_nanmean(eval_df['seg_mIoU']) if len(eval_df) else math.nan,
        'mean_frame_IoU_background_eval_only': safe_nanmean(eval_df['seg_IoU_background']) if len(eval_df) else math.nan,
        'mean_frame_IoU_road_eval_only': safe_nanmean(eval_df['seg_IoU_road']) if len(eval_df) else math.nan,
        'global_mIoU_from_total_hist_eval_only': float(np.nanmean(seg_ious_total)) if len(eval_df) else math.nan,
        'global_IoU_background_eval_only': float(seg_ious_total[0]) if len(seg_ious_total) > 0 and len(eval_df) else math.nan,
        'global_IoU_road_eval_only': float(seg_ious_total[1]) if len(seg_ious_total) > 1 and len(eval_df) else math.nan,
        'cls_top1_eval_only': float(cls_top1_from_cm(cls_cm)),
        'timestamp_match_tol_s': float(TIMESTAMP_MATCH_TOL),
        'img_dir': IMG_DIR,
        'mask_dir': MASK_DIR,
        'cls_ann_path': CLS_ANN_PATH,
        'config': CONFIG,
        'checkpoint': resolved_ckpt,
        'output_xlsx': OUTPUT_XLSX,
    }

    return df, eval_df, cls_cm, summary


def style_sheet_basic(xlsx_path: str):
    wb = load_workbook(xlsx_path)

    header_fill = PatternFill(fill_type='solid', fgColor='1F4E78')
    header_font = Font(color='FFFFFF', bold=True)
    center = Alignment(horizontal='center', vertical='center')

    for ws in wb.worksheets:
        ws.freeze_panes = 'A2'
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = center

        for col_idx, col_cells in enumerate(ws.columns, start=1):
            max_len = 0
            for c in col_cells:
                try:
                    v = '' if c.value is None else str(c.value)
                except Exception:
                    v = ''
                max_len = max(max_len, len(v))
            ws.column_dimensions[get_column_letter(col_idx)].width = min(max_len + 2, 45)
    wb.save(xlsx_path)


def write_excel(df: pd.DataFrame, eval_df: pd.DataFrame, cls_cm: np.ndarray, summary: Dict, out_path: str):
    ensure_dir_for_file(out_path)

    cm_df = pd.DataFrame(cls_cm, index=MODEL_CLS_NAMES, columns=MODEL_CLS_NAMES)
    summary_df = pd.DataFrame([summary])

    with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='PerFrame', index=False)
        eval_df.to_excel(writer, sheet_name='EvaluatedOnly', index=False)
        summary_df.to_excel(writer, sheet_name='Summary', index=False)
        cm_df.to_excel(writer, sheet_name='ConfusionMatrix')

    style_sheet_basic(out_path)


def main():
    df, eval_df, cls_cm, summary = evaluate_sequence()
    write_excel(df, eval_df, cls_cm, summary, OUTPUT_XLSX)

    print('\n========== RESUMEN ==========')
    for k, v in summary.items():
        print(f'{k}: {v}')
    print(f'\nExcel guardado en:\n{OUTPUT_XLSX}')


main()
